In [1]:
import io
import zipfile
import requests
import frontmatter

In [2]:
def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data


In [3]:
zoomcamp = read_repo_data('DataTalksClub', 'data-engineering-zoomcamp')
print(f"Found {len(zoomcamp)} documents")

Found 143 documents


In [4]:
import re

def split_markdown_by_level(text, level=2):
    """
    Split markdown text by a specific header level.
    
    :param text: Markdown text as a string
    :param level: Header level to split on
    :return: List of sections as strings
    """
    # This regex matches markdown headers
    # For level 2, it matches lines starting with "## "
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)

    # Split and keep the headers
    parts = pattern.split(text)
    
    sections = []
    for i in range(1, len(parts), 3):
        # We step by 3 because regex.split() with
        # capturing groups returns:
        # [before_match, group1, group2, after_match, ...]
        # here group1 is "## ", group2 is the header text
        header = parts[i] + parts[i+1]  # "## " + "Title"
        header = header.strip()

        # Get the content after this header
        content = ""
        if i+2 < len(parts):
            content = parts[i+2].strip()

        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header
        sections.append(section)
    
    return sections


In [5]:
zoomcamp_chunks = []

for doc in zoomcamp:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    sections = split_markdown_by_level(doc_content, level=2)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        zoomcamp_chunks.append(section_doc)

In [6]:
print(len(zoomcamp_chunks))
print(zoomcamp_chunks[0].keys())

620
dict_keys(['filename', 'section'])


In [7]:
from minsearch import Index

index = Index(
    text_fields=['section'],  # what fields do you have?
    keyword_fields=[]
)
index.fit(zoomcamp_chunks)

In [8]:
query = 'how do I setup Docker?'
results = index.search(query)
print(f"Number of results: {len(results)}")
print(results[0])

Number of results: 10
{'filename': 'data-engineering-zoomcamp-main/04-analytics-engineering/README.md', 'section': '## Setting up your environment\n\nChoose your setup path:\n\n### 🏠 [Local Setup](setup/local_setup.md)\n\n- **Stack**: DuckDB + dbt Core\n- **Cost**: Free\n- [→ Get Started](setup/local_setup.md)\n\n### ☁️ [Cloud Setup](setup/cloud_setup.md)\n\n- **Stack**: BigQuery + dbt Cloud\n- **Cost**: Free tier available (dbt Cloud Developer), BigQuery costs vary\n- **Requires**: Completed Module 3 with BigQuery data\n- [→ Get Started](setup/cloud_setup.md)'}


In [9]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
from minsearch import VectorSearch
from tqdm.auto import tqdm
import numpy as np

zcamp_embeddings = []

for record in tqdm(zoomcamp_chunks):
    text = record['section']
    v_doc = embedding_model.encode(text)
    zcamp_embeddings.append(v_doc)

zcamp_embeddings = np.array(zcamp_embeddings)

zcamp_vindex = VectorSearch()
zcamp_vindex.fit(zcamp_embeddings, zoomcamp_chunks)

  0%|          | 0/620 [00:00<?, ?it/s]

In [11]:
query = 'how do I setup Docker?'

# text search
text_results = index.search(query)
print("TEXT SEARCH:")
print(text_results[0]['section'][:200])

# vector search
q = embedding_model.encode(query)
vector_results = zcamp_vindex.search(q)
print("\nVECTOR SEARCH:")
print(vector_results[0]['section'])

TEXT SEARCH:
## Setting up your environment

Choose your setup path:

### 🏠 [Local Setup](setup/local_setup.md)

- **Stack**: DuckDB + dbt Core
- **Cost**: Free
- [→ Get Started](setup/local_setup.md)

### ☁️ [Clo

VECTOR SEARCH:
## Basic Docker Commands

Check Docker version:

```bash
docker --version
```

Run a simple container:

```bash
docker run hello-world
```

Run something more complex:

```bash
docker run ubuntu
```

Nothing happens. Need to run it in `-it` mode:

```bash
docker run -it ubuntu
```

We don't have `python` there so let's install it:

```bash
apt update && apt install python3
python3 -V
```


In [21]:
from typing import List, Any

def text_search(query: str, num_results: int = 3):
    return index.search(query, num_results=num_results)

def vector_search(query: str, num_results: int = 3):
    q = embedding_model.encode(query)
    return zcamp_vindex.search(q, num_results=num_results)

def hybrid_search(query: str) -> List[Any]:
    """
    Search the Zoomcamp course sections for relevant information.
    Args:
        query (str): The search query string.
    Returns:
        List[Any]: A list of search results from the course sections.
    """
    text_results = text_search(query, num_results=3)
    vector_results = vector_search(query, num_results=3)
    
    seen_ids = set()
    combined_results = []
    for result in text_results + vector_results:
        key = result['filename'] + result['section']
        if key not in seen_ids:
            seen_ids.add(key)
            # Truncate section to avoid token limits
            r = result.copy()
            r['section'] = result['section'][:500]
            combined_results.append(r)
    
    return combined_results


In [22]:
query = 'how do I setup Docker?'
print(f"Text: {len(text_search(query))} results")
print(f"Vector: {len(vector_search(query))} results")
print(f"Hybrid: {len(hybrid_search(query))} results")

Text: 3 results
Vector: 3 results
Hybrid: 6 results


In [17]:
from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv('../.env')
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [25]:
from pydantic_ai import Agent
from pydantic_ai.models.groq import GroqModel

model = GroqModel('llama-3.1-8b-instant')

system_prompt = """
You are a helpful assistant for a course. 
Use the search tool to find relevant information before answering questions.
If the first search doesn't give enough information, try different search terms.
"""

agent = Agent(
    model=model,
    system_prompt=system_prompt,
    tools=[hybrid_search]
)

In [23]:
question = "How long does it take to complete the course?"
result = await agent.run(user_prompt=question)
print(result.output)

The course length could not be found using the provided search tool. Is there anything else I can help you with?


In [26]:
question = "What tools do I need to install for module 1?"
result = await agent.run(user_prompt=question)
print(result.output)

For module 1, you need to install Docker and Python on your machine. It's also helpful but not required to have basic SQL knowledge. Additionally, it's recommended to have a basic understanding of Python.
